# 활성화 유저의 후속 PING 발신 리텐션 분석

## 분석 목적

신규 유저가 서비스의 핵심 보상인 **첫 PING 수신 + 힌트 열람**을 경험한 뒤에도 PING을 다시 보내는지 확인한다.  
이 노트북은 먼저 활성화 유저의 후속 발신 패턴을 탐색하고, 마지막에 대표 가설을 검증할 수 있는 비교 설계까지 정리한다.

> 대표 가설: 첫 수신과 힌트 열람을 경험한 유저는 그렇지 않은 유저보다 후속 PING 발신율이 높을 것이다.

### 분석 범위와 용어

| 항목 | 정의 |
|---|---|
| 데이터 기준일 | 2023-07-31 |
| 코호트 기준 | `signup_date` |
| 활성화 유저 | `mart_activation.is_activated = TRUE` |
| 후속 행동 | 가입 당일(D+0)을 제외한 PING 재발신 |
| 관찰 구간 | D+1, D+1~3, D+1~7, D+1~10 누적 |
| 관찰 가능 조건 | D+10까지 완전히 관찰 가능한 가입자만 포함 |

> **해석 주의:** 여기서 리텐션은  **후속 PING 발신 여부로 측정한 행동 리텐션**이다. 따라서 결과는 “재방문”보다 “재발신”으로 표현한다.


## 0. 실행 환경


In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

## 1. 활성화 유저의 누적 재발신율

활성화 유저가 가입 다음 날부터 D+1·D+3·D+7·D+10 이내에 PING을 한 번이라도 다시 보냈는지 확인한다. 모든 시점은 서로 배타적인 일별 리텐션이 아니라 **누적 재발신율**이다.


In [ ]:
%%bigquery df_retention_summary --project your-gcp-project
-- 활성화 유저의 가입일 기준 누적 재발신율
-- D+0 제외, D+1부터 포함
-- 관찰 종료일: 2023-07-31

WITH params AS (
  SELECT DATE '2023-07-31' AS observation_end_date
),

activated_users AS (
  SELECT
    activation.user_id,
    activation.signup_date
  FROM `your-gcp-project.mart.mart_activation` AS activation
  CROSS JOIN params
  WHERE activation.is_activated
    -- D+10까지 완전히 관찰 가능한 가입자만 포함
    AND activation.signup_date
        <= DATE_SUB(params.observation_end_date, INTERVAL 10 DAY)
),

send_dates AS (
  SELECT DISTINCT
    vote.sender_user_id AS user_id,
    DATE(vote.voted_at) AS send_date
  FROM `your-gcp-project.stage.stg_vote` AS vote
  CROSS JOIN params
  WHERE DATE(vote.voted_at) <= params.observation_end_date
),

user_retention_flags AS (
  SELECT
    activated.user_id,
    activated.signup_date,

    -- D+1: 가입 다음 날 발신
    COUNTIF(
      send.send_date <= DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
    ) > 0 AS retained_within_d1,

    -- D+1~D+3 중 한 번이라도 발신
    COUNTIF(
      send.send_date <= DATE_ADD(activated.signup_date, INTERVAL 3 DAY)
    ) > 0 AS retained_within_d3,

    -- D+1~D+7 중 한 번이라도 발신
    COUNTIF(
      send.send_date <= DATE_ADD(activated.signup_date, INTERVAL 7 DAY)
    ) > 0 AS retained_within_d7,

    -- D+1~D+10 중 한 번이라도 발신
    COUNTIF(
      send.send_date <= DATE_ADD(activated.signup_date, INTERVAL 10 DAY)
    ) > 0 AS retained_within_d10

  FROM activated_users AS activated

  LEFT JOIN send_dates AS send
    ON activated.user_id = send.user_id
   -- 가입 당일 D+0 제외
   AND send.send_date > activated.signup_date
   -- 필요한 최대 범위까지만 조인
   AND send.send_date
       <= DATE_ADD(activated.signup_date, INTERVAL 10 DAY)

  GROUP BY
    activated.user_id,
    activated.signup_date
)

SELECT
  COUNT(*) AS activated_total,

  COUNTIF(retained_within_d1) AS d1_users,
  ROUND(
    SAFE_DIVIDE(COUNTIF(retained_within_d1), COUNT(*)) * 100,
    1
  ) AS d1_rate_pct,

  COUNTIF(retained_within_d3) AS d3_users,
  ROUND(
    SAFE_DIVIDE(COUNTIF(retained_within_d3), COUNT(*)) * 100,
    1
  ) AS d3_rate_pct,

  COUNTIF(retained_within_d7) AS d7_users,
  ROUND(
    SAFE_DIVIDE(COUNTIF(retained_within_d7), COUNT(*)) * 100,
    1
  ) AS d7_rate_pct,

  COUNTIF(retained_within_d10) AS d10_users,
  ROUND(
    SAFE_DIVIDE(COUNTIF(retained_within_d10), COUNT(*)) * 100,
    1
  ) AS d10_rate_pct

FROM user_retention_flags;

In [ ]:
df_retention_summary


### 결과 해석

| 관찰 시점 | 누적 재발신 유저 | 누적 재발신율 |
|---|---:|---:|
| D+1 이내 | 3,137명 | 62.0% |
| D+3 이내 | 3,339명 | 66.0% |
| D+7 이내 | 3,393명 | 67.1% |
| D+10 이내 | 3,412명 | 67.4% |

- 활성화 유저 5,059명 중 3,412명(67.4%)이 D+10 이내에 PING을 다시 발신했다.
- 최초 재발신은 D+1에 강하게 집중되어 있다. D+1 이후 새롭게 재발신한 유저는 D+2~3에 202명, D+4~7에 54명, D+8~10에 19명뿐이다.
- D+10 이내 재발신자 3,412명 중 3,137명(약 92.0%)이 가입 다음 날까지 재발신했다.
- 따라서 활성화 이후 반복 행동을 유도하려면 **가입 다음 날까지의 개입**이 우선순위가 될 가능성이 높다.

> 이 결과만으로 “수신과 힌트 경험이 재발신을 높였다”고 결론 내릴 수는 없다. 현재 분석 대상이 활성화 유저로 한정되어 있어 비교군이 없기 때문이다.


## 2. 네트워크 규모별 D+7 재발신율

서비스는 같은 학교 유저 40명 이상, 같은 학급 유저 4명 이상이어야 질문 세트가 실행되는 네트워크형 서비스다. 이에 가입 당시의 학교·학급 가입자 규모와 활성화 유저의 D+7 재발신율이 함께 증가하는지 탐색한다.

> 이 단계는 임계값 후보를 찾는 EDA다. 구간별 차이는 연관성일 뿐, 네트워크 규모의 인과 효과로 해석하지 않는다.


## 가입 당시 학교 규모별 D+7 재발신율

In [ ]:
%%bigquery df_retention_by_school_size --project your-gcp-project

-- 가입 당시 학교 가입자 규모별 D+7 누적 후속 PING 발신율

WITH params AS (
  SELECT DATE '2023-07-31' AS observation_end_date
),

activated_users AS (
  SELECT
    activation.user_id,
    activation.signup_date,
    activation.school_signup_count_asof_signup
  FROM `your-gcp-project.mart.mart_activation` AS activation
  CROSS JOIN params
  WHERE activation.is_activated
    AND activation.signup_date
      <= DATE_SUB(params.observation_end_date, INTERVAL 10 DAY)
),

send_dates AS (
  SELECT DISTINCT
    vote.sender_user_id AS user_id,
    DATE(vote.voted_at) AS send_date
  FROM `your-gcp-project.stage.stg_vote` AS vote
  CROSS JOIN params
  WHERE DATE(vote.voted_at) <= params.observation_end_date
),

user_retention_flags AS (
  SELECT
    activated.user_id,
    activated.school_signup_count_asof_signup,

    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(activated.signup_date, INTERVAL 7 DAY)
    ) > 0 AS retained_within_d7

  FROM activated_users AS activated

  LEFT JOIN send_dates AS send
    ON activated.user_id = send.user_id
   AND send.send_date
       BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
           AND DATE_ADD(activated.signup_date, INTERVAL 7 DAY)

  GROUP BY
    activated.user_id,
    activated.school_signup_count_asof_signup
),

bucketed AS (
  SELECT
    user_id,
    retained_within_d7,

    CASE
      WHEN school_signup_count_asof_signup < 100
        THEN '01_40~99명'
      WHEN school_signup_count_asof_signup < 200
        THEN '02_100~199명'
      WHEN school_signup_count_asof_signup < 300
        THEN '03_200~299명'
      WHEN school_signup_count_asof_signup < 400
        THEN '04_300~399명'
      ELSE '05_400명 이상'
    END AS school_size_group

  FROM user_retention_flags
)

SELECT
  school_size_group,
  COUNT(*) AS activated_users,
  COUNTIF(retained_within_d7) AS d7_resend_users,

  ROUND(
    SAFE_DIVIDE(
      COUNTIF(retained_within_d7),
      COUNT(*)
    ) * 100,
    2
  ) AS d7_resend_rate_pct

FROM bucketed
GROUP BY school_size_group
ORDER BY school_size_group;

In [ ]:
df_retention_by_school_size

### 학교 규모 결과

- 가입 당시 학교 가입자가 40~99명인 구간의 D+7 재발신율은 39.71%, 400명 이상 구간은 93.88%로 나타났다.
- 학교 가입자 규모가 큰 구간일수록 재발신율이 일관되게 높았다.
- 다만 학교 규모에는 가입 시기, 학교 유형, 학년 구성 등 다른 조건이 함께 섞일 수 있으므로 이후 코호트·세그먼트 통제가 필요하다.


## 가입 당시 학급 규모별 D+7 재발신율

In [ ]:
%%bigquery df_retention_by_class_size --project your-gcp-project

-- 가입 당시 학급 가입자 규모별 D+7 누적 후속 PING 발신율

WITH params AS (
  SELECT DATE '2023-07-31' AS observation_end_date
),

activated_users AS (
  SELECT
    activation.user_id,
    activation.signup_date,
    activation.class_signup_count_asof_signup
  FROM `your-gcp-project.mart.mart_activation` AS activation
  CROSS JOIN params
  WHERE activation.is_activated
    AND activation.signup_date
      <= DATE_SUB(params.observation_end_date, INTERVAL 10 DAY)
),

send_dates AS (
  SELECT DISTINCT
    vote.sender_user_id AS user_id,
    DATE(vote.voted_at) AS send_date
  FROM `your-gcp-project.stage.stg_vote` AS vote
  CROSS JOIN params
  WHERE DATE(vote.voted_at) <= params.observation_end_date
),

user_retention_flags AS (
  SELECT
    activated.user_id,
    activated.class_signup_count_asof_signup,

    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(activated.signup_date, INTERVAL 7 DAY)
    ) > 0 AS retained_within_d7

  FROM activated_users AS activated

  LEFT JOIN send_dates AS send
    ON activated.user_id = send.user_id
   AND send.send_date
       BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
           AND DATE_ADD(activated.signup_date, INTERVAL 7 DAY)

  GROUP BY
    activated.user_id,
    activated.class_signup_count_asof_signup
),

bucketed AS (
  SELECT
    user_id,
    retained_within_d7,

    CASE
      WHEN class_signup_count_asof_signup < 10
        THEN '01_4~9명'
      WHEN class_signup_count_asof_signup < 15
        THEN '02_10~14명'
      WHEN class_signup_count_asof_signup < 20
        THEN '03_15~19명'
      WHEN class_signup_count_asof_signup < 25
        THEN '04_20~24명'
      ELSE '05_25명 이상'
    END AS class_size_group

  FROM user_retention_flags
)

SELECT
  class_size_group,
  COUNT(*) AS activated_users,
  COUNTIF(retained_within_d7) AS d7_resend_users,

  ROUND(
    SAFE_DIVIDE(
      COUNTIF(retained_within_d7),
      COUNT(*)
    ) * 100,
    2
  ) AS d7_resend_rate_pct

FROM bucketed
GROUP BY class_size_group
ORDER BY class_size_group;

In [ ]:
df_retention_by_class_size

### 학급 규모 결과

- 학급 가입자 4-9명 구간은 61.93%, 20-24명 구간은 85.34%로 학급 규모가 커질수록 대체로 재발신율이 높았다.
- 25명 이상 구간은 84.38%지만 표본이 32명으로 작아 20~24명 구간과의 미세한 차이는 의미 있게 해석하기 어렵다.
- 학교 규모 결과와 함께 보면, 활성화 이후 반복 발신에도 **주변 사용자 풀이 충분한 환경**이 중요할 가능성이 있다.


## 3. 사용자 세그먼트별 D+7 재발신율

성별·학년·학교 유형별 차이를 확인해 특정 집단에 결과가 치우쳐 있는지 탐색한다. 표본 30명 미만 그룹은 참고값으로만 사용한다.


In [ ]:
%%bigquery df_retention_by_user_segment --project your-gcp-project

-- 성별·학년·학교 유형별 D+7 누적 후속 PING 발신율

WITH params AS (
  SELECT DATE '2023-07-31' AS observation_end_date
),

activated_users AS (
  SELECT
    activation.user_id,
    activation.signup_date,
    activation.gender,
    activation.grade,
    activation.school_type
  FROM `your-gcp-project.mart.mart_activation` AS activation
  CROSS JOIN params
  WHERE activation.is_activated
    AND activation.signup_date
      <= DATE_SUB(params.observation_end_date, INTERVAL 10 DAY)
),

send_dates AS (
  SELECT DISTINCT
    vote.sender_user_id AS user_id,
    DATE(vote.voted_at) AS send_date
  FROM `your-gcp-project.stage.stg_vote` AS vote
  CROSS JOIN params
  WHERE DATE(vote.voted_at) <= params.observation_end_date
),

user_retention_flags AS (
  SELECT
    activated.user_id,
    activated.gender,
    activated.grade,
    activated.school_type,

    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(activated.signup_date, INTERVAL 7 DAY)
    ) > 0 AS retained_within_d7

  FROM activated_users AS activated

  LEFT JOIN send_dates AS send
    ON activated.user_id = send.user_id
   AND send.send_date
       BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
           AND DATE_ADD(activated.signup_date, INTERVAL 7 DAY)

  GROUP BY
    activated.user_id,
    activated.gender,
    activated.grade,
    activated.school_type
),

segments AS (
  SELECT
    'gender' AS segment_type,
    COALESCE(CAST(gender AS STRING), '미상') AS segment_value,
    user_id,
    retained_within_d7
  FROM user_retention_flags

  UNION ALL

  SELECT
    'grade' AS segment_type,
    COALESCE(CAST(grade AS STRING), '미상') AS segment_value,
    user_id,
    retained_within_d7
  FROM user_retention_flags

  UNION ALL

  SELECT
    'school_type' AS segment_type,
    COALESCE(CAST(school_type AS STRING), '미상') AS segment_value,
    user_id,
    retained_within_d7
  FROM user_retention_flags
)

SELECT
  segment_type,
  segment_value,
  COUNT(*) AS activated_users,
  COUNTIF(retained_within_d7) AS d7_resend_users,

  ROUND(
    SAFE_DIVIDE(
      COUNTIF(retained_within_d7),
      COUNT(*)
    ) * 100,
    2
  ) AS d7_resend_rate_pct,

  -- 표본이 너무 작은 그룹 확인
  COUNT(*) < 30 AS is_small_sample

FROM segments
GROUP BY
  segment_type,
  segment_value
ORDER BY
  segment_type,
  d7_resend_rate_pct DESC;

In [ ]:
df_retention_by_user_segment

### 세그먼트 결과

- 성별: M 69.39%, F 64.96%로 4.43%p 차이가 있다.
- 학년: 1학년 55.16%, 2학년 73.83%, 3학년 81.35%로 학년이 높을수록 재발신율이 높았다.
- 학교 유형: M 78.03%, H 64.72%로 13.31%p 차이가 있다.

학년과 학교 유형에 따른 차이가 크므로, 활성화 그룹 간 비교에서도 이 변수들의 분포 차이를 확인해야 한다. 단순 전체 평균만 비교하면 세그먼트 구성 차이가 활성화 효과처럼 보일 수 있다.


## 4. 사용자 단위 리텐션 마트 생성

반복 분석에 사용할 수 있도록 활성화 유저 1명당 1행의 리텐션 마트를 생성한다. 가입일로 파티셔닝하고 학교·학급 관련 키로 클러스터링해 후속 집계와 대시보드 조회 비용을 줄인다.


In [ ]:
%%bigquery --project your-gcp-project

CREATE OR REPLACE TABLE
  `your-gcp-project.mart.mart_activation_retention`

PARTITION BY signup_date
CLUSTER BY school_id, group_id

OPTIONS (
  description = '활성화 유저의 가입일 기준 D+1/D+3/D+7/D+10 누적 후속 PING 발신 여부'
)

AS

WITH params AS (
  SELECT DATE '2023-07-31' AS observation_end_date
),

activated_users AS (
  SELECT
    activation.user_id,
    activation.signup_at,
    activation.signup_date,

    activation.school_id,
    activation.school_type,
    activation.grade,
    activation.class_num,
    activation.group_id,
    activation.gender,

    activation.first_receive_at,
    activation.first_hint_opened_at,
    activation.receive_count,
    activation.time_to_hint_open_hours,
    activation.time_to_hint_open_bucket,

    activation.school_signup_count_asof_signup,
    activation.class_signup_count_asof_signup

  FROM `your-gcp-project.mart.mart_activation` AS activation
  CROSS JOIN params
  WHERE activation.is_activated
    AND activation.signup_date
      <= DATE_SUB(params.observation_end_date, INTERVAL 10 DAY)
),

send_dates AS (
  SELECT DISTINCT
    vote.sender_user_id AS user_id,
    DATE(vote.voted_at) AS send_date
  FROM `your-gcp-project.stage.stg_vote` AS vote
  CROSS JOIN params
  WHERE DATE(vote.voted_at) <= params.observation_end_date
),

retention_flags AS (
  SELECT
    activated.user_id,
    activated.signup_at,
    activated.signup_date,

    activated.school_id,
    activated.school_type,
    activated.grade,
    activated.class_num,
    activated.group_id,
    activated.gender,

    activated.first_receive_at,
    activated.first_hint_opened_at,
    activated.receive_count,
    activated.time_to_hint_open_hours,
    activated.time_to_hint_open_bucket,

    activated.school_signup_count_asof_signup,
    activated.class_signup_count_asof_signup,

    MIN(send.send_date) AS first_resend_date,

    DATE_DIFF(
      MIN(send.send_date),
      activated.signup_date,
      DAY
    ) AS days_to_first_resend,

    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
    ) > 0 AS retained_within_d1,

    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(activated.signup_date, INTERVAL 3 DAY)
    ) > 0 AS retained_within_d3,

    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(activated.signup_date, INTERVAL 7 DAY)
    ) > 0 AS retained_within_d7,

    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(activated.signup_date, INTERVAL 10 DAY)
    ) > 0 AS retained_within_d10

  FROM activated_users AS activated

  LEFT JOIN send_dates AS send
    ON activated.user_id = send.user_id
   AND send.send_date
       BETWEEN DATE_ADD(activated.signup_date, INTERVAL 1 DAY)
           AND DATE_ADD(activated.signup_date, INTERVAL 10 DAY)

  GROUP BY
    activated.user_id,
    activated.signup_at,
    activated.signup_date,
    activated.school_id,
    activated.school_type,
    activated.grade,
    activated.class_num,
    activated.group_id,
    activated.gender,
    activated.first_receive_at,
    activated.first_hint_opened_at,
    activated.receive_count,
    activated.time_to_hint_open_hours,
    activated.time_to_hint_open_bucket,
    activated.school_signup_count_asof_signup,
    activated.class_signup_count_asof_signup
)

SELECT
  *,

  CASE
    WHEN days_to_first_resend = 1
      THEN '01_D+1'
    WHEN days_to_first_resend BETWEEN 2 AND 3
      THEN '02_D+2~3'
    WHEN days_to_first_resend BETWEEN 4 AND 7
      THEN '03_D+4~7'
    WHEN days_to_first_resend BETWEEN 8 AND 10
      THEN '04_D+8~10'
    ELSE '05_D+10이내_미재발신'
  END AS first_resend_period

FROM retention_flags;

### 4-1. 리텐션 마트 품질 점검

다음을 확인한다.

- 전체 행 수와 고유 사용자 수가 같은가
- 기존 EDA의 D+1·3·7·10 재발신자 수와 일치하는가
- 누적 지표의 포함 관계(`D1 ⊆ D3 ⊆ D7 ⊆ D10`)를 위반하는 행이 없는가


In [ ]:
%%bigquery df_retention_mart_qc --project your-gcp-project

SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT user_id) AS distinct_users,

  COUNTIF(retained_within_d1) AS d1_users,
  COUNTIF(retained_within_d3) AS d3_users,
  COUNTIF(retained_within_d7) AS d7_users,
  COUNTIF(retained_within_d10) AS d10_users,

  COUNTIF(retained_within_d1 AND NOT retained_within_d3)
    AS invalid_d1_d3,

  COUNTIF(retained_within_d3 AND NOT retained_within_d7)
    AS invalid_d3_d7,

  COUNTIF(retained_within_d7 AND NOT retained_within_d10)
    AS invalid_d7_d10

FROM `your-gcp-project.mart.mart_activation_retention`;

In [ ]:
df_retention_mart_qc

### 마트 QC 결과

- 전체 5,059행 = 고유 사용자 5,059명으로 사용자 단위 유일성이 확인됐다.
- D+1 3,137명, D+3 3,339명, D+7 3,393명, D+10 3,412명으로 앞선 EDA 결과와 일치한다.
- 누적 구간 포함 관계 위반은 모두 0건이다.


## 5. 전체 활성화 여부별 비교 — 현황 참고용

아래 쿼리는 전체 가입자를 `is_activated`로 나눈 단순 비교다. 활성화 유저 5,059명과 미활성화 유저 659,650명의 D+1·3·7·10 재발신율 차이를 보여준다.

> ⚠️ **대표 가설의 최종 검증으로 사용하지 않는다.** 미활성화 그룹에는 가입 후 PING을 한 번도 보내지 않은 유저가 대량 포함되어 출발 행동과 서비스 이용 의지가 완전히 다르다. 또한 `is_activated`가 관찰 기간 전체의 경험을 사용한다면, 결과 기간 이후의 수신·힌트가 활성화 판정에 들어가는 시간 누수 가능성도 있다.


In [ ]:
%%bigquery df_retention_by_activation_status --project your-gcp-project

-- 활성화 여부별 D+1/D+3/D+7/D+10 누적 후속 PING 발신율 비교
-- D+0 제외
-- 관찰 종료일: 2023-07-31

WITH params AS (
  SELECT DATE '2023-07-31' AS observation_end_date
),

eligible_users AS (
  SELECT
    activation.user_id,
    activation.signup_date,
    activation.is_activated
  FROM `your-gcp-project.mart.mart_activation` AS activation
  CROSS JOIN params
  WHERE activation.signup_date
    <= DATE_SUB(params.observation_end_date, INTERVAL 10 DAY)
),

send_dates AS (
  SELECT DISTINCT
    vote.sender_user_id AS user_id,
    DATE(vote.voted_at) AS send_date
  FROM `your-gcp-project.stage.stg_vote` AS vote
  CROSS JOIN params
  WHERE DATE(vote.voted_at) <= params.observation_end_date
),

user_retention_flags AS (
  SELECT
    eligible.user_id,
    eligible.is_activated,

    -- D+1 누적 재발신 여부
    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(eligible.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(eligible.signup_date, INTERVAL 1 DAY)
    ) > 0 AS retained_within_d1,

    -- D+1~D+3 누적 재발신 여부
    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(eligible.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(eligible.signup_date, INTERVAL 3 DAY)
    ) > 0 AS retained_within_d3,

    -- D+1~D+7 누적 재발신 여부
    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(eligible.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(eligible.signup_date, INTERVAL 7 DAY)
    ) > 0 AS retained_within_d7,

    -- D+1~D+10 누적 재발신 여부
    COUNTIF(
      send.send_date
        BETWEEN DATE_ADD(eligible.signup_date, INTERVAL 1 DAY)
            AND DATE_ADD(eligible.signup_date, INTERVAL 10 DAY)
    ) > 0 AS retained_within_d10

  FROM eligible_users AS eligible

  LEFT JOIN send_dates AS send
    ON eligible.user_id = send.user_id
   AND send.send_date
       BETWEEN DATE_ADD(eligible.signup_date, INTERVAL 1 DAY)
           AND DATE_ADD(eligible.signup_date, INTERVAL 10 DAY)

  GROUP BY
    eligible.user_id,
    eligible.is_activated
),

summary AS (
  SELECT
    is_activated,
    COUNT(*) AS users,

    COUNTIF(retained_within_d1) AS d1_resend_users,
    SAFE_DIVIDE(
      COUNTIF(retained_within_d1),
      COUNT(*)
    ) * 100 AS d1_resend_rate_pct,

    COUNTIF(retained_within_d3) AS d3_resend_users,
    SAFE_DIVIDE(
      COUNTIF(retained_within_d3),
      COUNT(*)
    ) * 100 AS d3_resend_rate_pct,

    COUNTIF(retained_within_d7) AS d7_resend_users,
    SAFE_DIVIDE(
      COUNTIF(retained_within_d7),
      COUNT(*)
    ) * 100 AS d7_resend_rate_pct,

    COUNTIF(retained_within_d10) AS d10_resend_users,
    SAFE_DIVIDE(
      COUNTIF(retained_within_d10),
      COUNT(*)
    ) * 100 AS d10_resend_rate_pct

  FROM user_retention_flags
  GROUP BY is_activated
),

comparison AS (
  SELECT
    MAX(IF(NOT is_activated, d1_resend_rate_pct, NULL))
      AS non_activated_d1_rate,

    MAX(IF(NOT is_activated, d3_resend_rate_pct, NULL))
      AS non_activated_d3_rate,

    MAX(IF(NOT is_activated, d7_resend_rate_pct, NULL))
      AS non_activated_d7_rate,

    MAX(IF(NOT is_activated, d10_resend_rate_pct, NULL))
      AS non_activated_d10_rate

  FROM summary
)

SELECT
  CASE
    WHEN summary.is_activated THEN '활성화'
    ELSE '미활성화'
  END AS activation_status,

  summary.users,

  -- D+1
  summary.d1_resend_users,
  ROUND(summary.d1_resend_rate_pct, 2) AS d1_resend_rate_pct,
  ROUND(
    summary.d1_resend_rate_pct - comparison.non_activated_d1_rate,
    2
  ) AS d1_difference_pp,
  ROUND(
    SAFE_DIVIDE(
      summary.d1_resend_rate_pct,
      comparison.non_activated_d1_rate
    ),
    2
  ) AS d1_rate_ratio,

  -- D+3
  summary.d3_resend_users,
  ROUND(summary.d3_resend_rate_pct, 2) AS d3_resend_rate_pct,
  ROUND(
    summary.d3_resend_rate_pct - comparison.non_activated_d3_rate,
    2
  ) AS d3_difference_pp,
  ROUND(
    SAFE_DIVIDE(
      summary.d3_resend_rate_pct,
      comparison.non_activated_d3_rate
    ),
    2
  ) AS d3_rate_ratio,

  -- D+7
  summary.d7_resend_users,
  ROUND(summary.d7_resend_rate_pct, 2) AS d7_resend_rate_pct,
  ROUND(
    summary.d7_resend_rate_pct - comparison.non_activated_d7_rate,
    2
  ) AS d7_difference_pp,
  ROUND(
    SAFE_DIVIDE(
      summary.d7_resend_rate_pct,
      comparison.non_activated_d7_rate
    ),
    2
  ) AS d7_rate_ratio,

  -- D+10
  summary.d10_resend_users,
  ROUND(summary.d10_resend_rate_pct, 2) AS d10_resend_rate_pct,
  ROUND(
    summary.d10_resend_rate_pct - comparison.non_activated_d10_rate,
    2
  ) AS d10_difference_pp,
  ROUND(
    SAFE_DIVIDE(
      summary.d10_resend_rate_pct,
      comparison.non_activated_d10_rate
    ),
    2
  ) AS d10_rate_ratio

FROM summary
CROSS JOIN comparison
ORDER BY summary.is_activated DESC;

In [ ]:
df_retention_by_activation_status

### 단순 비교 결과의 올바른 해석

- 활성화 그룹은 모든 시점에서 미활성화 그룹보다 재발신율이 매우 높게 나타났다.
- 그러나 두 그룹의 표본과 초기 행동이 지나치게 다르므로 367~466배의 비율을 “활성화 경험의 효과”라고 해석하면 안 된다.
- 이 결과는 **활성화 상태와 후속 발신 간 강한 연관성이 관찰됐다**는 현황 확인까지만 사용한다.


## 6. 대표 가설 검증 설계

### 6-1. 시간축과 비교군

활성화 경험이 결과보다 먼저 발생하도록 노출 기간과 결과 기간을 분리한다.

| 구분 | 가입 당일(D0) 경험 | 의미 |
|---|---|---|
| G0 | 발신 O, 수신 X | 핵심 보상 미경험 |
| G1 | 발신 O, 수신 O, 힌트 X | 수신까지 경험 |
| G2 | 발신 O, 수신 O, 힌트 O | 수신과 힌트 모두 경험 |
| 결과 | D+1·3·7·10 누적 재발신 | D0 이후의 후속 행동 |

전체 가입자가 아니라 **D0에 최소 1회 PING을 보낸 유저만** 비교한다. 이렇게 해야 세 그룹 모두 최소한의 초기 발신 행동을 했다는 공통 출발선을 갖는다.

- **주 비교: G2 vs G1** — 둘 다 수신했으며 힌트 열람 여부가 다르므로 대표 가설에 가장 직접적이다.
- **보조 비교: G2 vs G0** — 전체 핵심 보상 경험 완료 여부의 차이를 본다.
- **추가 확인: G1 vs G0** — 수신 경험 자체와 후속 발신의 관계를 본다.


In [ ]:
%%bigquery df_retention_by_d0_experience --project your-gcp-project

-- 대표 가설 검증용 비교군
-- 노출: 가입 당일(D0)의 수신·힌트 경험
-- 결과: D+1/D+3/D+7/D+10 누적 후속 PING 발신
-- 관찰 종료일: 2023-07-31
-- D+0 발신은 비교군 선정에만 사용하고 리텐션 계산에서는 제외

WITH params AS (
  SELECT DATE '2023-07-31' AS observation_end_date
),

eligible_users AS (
  SELECT
    activation.user_id,
    activation.signup_at,
    activation.signup_date,
    activation.first_receive_at,
    activation.first_hint_opened_at
  FROM `your-gcp-project.mart.mart_activation` AS activation
  CROSS JOIN params
  -- D+10까지 완전히 관찰할 수 있는 가입자만 포함
  WHERE activation.signup_date
    <= DATE_SUB(params.observation_end_date, INTERVAL 10 DAY)
),

send_dates AS (
  SELECT DISTINCT
    vote.sender_user_id AS user_id,
    DATE(vote.voted_at) AS send_date
  FROM `your-gcp-project.stage.stg_vote` AS vote
  CROSS JOIN params
  WHERE DATE(vote.voted_at) <= params.observation_end_date
),

user_flags AS (
  SELECT
    eligible.user_id,
    eligible.signup_date,

    -- 비교군 공통 조건: 가입 당일 최소 1회 발신
    COUNTIF(
      send.send_date = eligible.signup_date
    ) > 0 AS sent_on_d0,

    -- 가입 당일 수신 여부
    -- 수신 기록이 없으면 NULL이 아닌 FALSE로 처리
    COALESCE(
      eligible.first_receive_at >= eligible.signup_at
      AND DATE(eligible.first_receive_at) = eligible.signup_date,
      FALSE
    ) AS received_on_d0,

    -- 가입 당일 힌트 열람 여부
    -- 힌트 열람 기록이 없으면 NULL이 아닌 FALSE로 처리
    COALESCE(
      eligible.first_hint_opened_at >= eligible.signup_at
      AND DATE(eligible.first_hint_opened_at) = eligible.signup_date,
      FALSE
    ) AS hint_opened_on_d0,

    -- D+1 당일 재발신
    COUNTIF(
      send.send_date = DATE_ADD(
        eligible.signup_date,
        INTERVAL 1 DAY
      )
    ) > 0 AS retained_within_d1,

    -- D+1부터 D+3까지 누적 재발신
    COUNTIF(
      send.send_date BETWEEN
        DATE_ADD(eligible.signup_date, INTERVAL 1 DAY)
        AND DATE_ADD(eligible.signup_date, INTERVAL 3 DAY)
    ) > 0 AS retained_within_d3,

    -- D+1부터 D+7까지 누적 재발신
    COUNTIF(
      send.send_date BETWEEN
        DATE_ADD(eligible.signup_date, INTERVAL 1 DAY)
        AND DATE_ADD(eligible.signup_date, INTERVAL 7 DAY)
    ) > 0 AS retained_within_d7,

    -- D+1부터 D+10까지 누적 재발신
    COUNTIF(
      send.send_date BETWEEN
        DATE_ADD(eligible.signup_date, INTERVAL 1 DAY)
        AND DATE_ADD(eligible.signup_date, INTERVAL 10 DAY)
    ) > 0 AS retained_within_d10

  FROM eligible_users AS eligible
  LEFT JOIN send_dates AS send
    ON eligible.user_id = send.user_id
    AND send.send_date BETWEEN
      eligible.signup_date
      AND DATE_ADD(eligible.signup_date, INTERVAL 10 DAY)
  GROUP BY
    eligible.user_id,
    eligible.signup_at,
    eligible.signup_date,
    eligible.first_receive_at,
    eligible.first_hint_opened_at
),

grouped AS (
  SELECT
    user_id,
    CASE
      WHEN NOT received_on_d0
        THEN 'G0_D0발신_수신없음'
      WHEN received_on_d0
        AND NOT hint_opened_on_d0
        THEN 'G1_D0수신_힌트미열람'
      WHEN received_on_d0
        AND hint_opened_on_d0
        THEN 'G2_D0수신+힌트'
    END AS experience_group,
    retained_within_d1,
    retained_within_d3,
    retained_within_d7,
    retained_within_d10
  FROM user_flags
  -- 가입 당일 발신한 사용자만 비교
  WHERE sent_on_d0
)

SELECT
  experience_group,
  COUNT(*) AS users,

  COUNTIF(retained_within_d1) AS d1_resend_users,
  ROUND(
    100 * SAFE_DIVIDE(
      COUNTIF(retained_within_d1),
      COUNT(*)
    ),
    2
  ) AS d1_resend_rate_pct,

  COUNTIF(retained_within_d3) AS d3_resend_users,
  ROUND(
    100 * SAFE_DIVIDE(
      COUNTIF(retained_within_d3),
      COUNT(*)
    ),
    2
  ) AS d3_resend_rate_pct,

  COUNTIF(retained_within_d7) AS d7_resend_users,
  ROUND(
    100 * SAFE_DIVIDE(
      COUNTIF(retained_within_d7),
      COUNT(*)
    ),
    2
  ) AS d7_resend_rate_pct,

  COUNTIF(retained_within_d10) AS d10_resend_users,
  ROUND(
    100 * SAFE_DIVIDE(
      COUNTIF(retained_within_d10),
      COUNT(*)
    ),
    2
  ) AS d10_resend_rate_pct

FROM grouped
GROUP BY experience_group
ORDER BY experience_group;

In [ ]:
df_retention_by_d0_experience


### 6-2. 효과 크기와 통계 검정

가설 판단에서는 재발신율만 보지 않고 다음을 함께 본다.

1. 그룹별 표본 수
2. 절대 차이(%p): `G2 재발신율 - 비교 그룹 재발신율`
3. 상대비: `G2 재발신율 / 비교 그룹 재발신율`
4. 두 비율 z-검정의 p-value와 다중 비교 보정 결과

아래 코드는 D+1·3·7·10에서 G2와 G1, G2와 G0를 비교한다. 한 번에 여러 시점을 검정하므로 Holm 방식으로 p-value를 보정한다.


In [ ]:
import pandas as pd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportions_ztest


def compare_retention_groups(summary_df, treatment_group, control_group):
    """시점별 두 그룹의 재발신율 차이와 비율 검정 결과를 반환한다."""
    indexed = summary_df.set_index("experience_group")
    treatment = indexed.loc[treatment_group]
    control = indexed.loc[control_group]

    results = []
    for day in (1, 3, 7, 10):
        resend_col = f"d{day}_resend_users"
        rate_col = f"d{day}_resend_rate_pct"

        counts = [treatment[resend_col], control[resend_col]]
        totals = [treatment["users"], control["users"]]
        _, p_value = proportions_ztest(count=counts, nobs=totals)

        treatment_rate = treatment[rate_col]
        control_rate = control[rate_col]
        results.append(
            {
                "comparison": f"{treatment_group} vs {control_group}",
                "day": f"D+{day}",
                "treatment_rate_pct": treatment_rate,
                "control_rate_pct": control_rate,
                "difference_pp": treatment_rate - control_rate,
                "rate_ratio": (
                    treatment_rate / control_rate if control_rate > 0 else None
                ),
                "p_value": p_value,
            }
        )

    result_df = pd.DataFrame(results)
    result_df["p_value_holm"] = multipletests(
        result_df["p_value"], method="holm"
    )[1]
    result_df["significant_0_05"] = result_df["p_value_holm"] < 0.05
    return result_df


primary_test = compare_retention_groups(
    df_retention_by_d0_experience,
    treatment_group="G2_D0수신+힌트",
    control_group="G1_D0수신_힌트미열람",
)

secondary_test = compare_retention_groups(
    df_retention_by_d0_experience,
    treatment_group="G2_D0수신+힌트",
    control_group="G0_D0발신_수신없음",
)

pd.concat([primary_test, secondary_test], ignore_index=True).round(4)


### 6-3. 가설 판단 기준

- **가설 지지:** G대표 지표인 D+7에서 G2의 재발신율이 G1보다 높고, Holm 보정 p-value가 0.05 미만이며, 퍼센트포인트 차이도 실무적으로 의미 있는 경우. D+1·D+3·D+10에서도 같은 방향이 나타나는지 함께 확인 필요
- **부분 지지:** D+7 차이가 통계적으로 유의하지 않거나 효과 크기가 작지만, 일부 시점에서만 일관된 차이가 확인되는 경우
- **기각 또는 보류:** G2와 G1의 차이가 거의 없거나, 방향이 시점마다 다르거나, 보정 p-value가 0.05 이상이라 차이를 확인할 근거가 부족한 경우

통계적 유의성만으로 결론 내리지 않고 퍼센트포인트 차이, 표본 수, 시점별 방향의 일관성을 함께 판단한다. 표본이 크면 실무적으로 작은 차이도 통계적으로 유의하게 나타날 수 있기 때문이다.

### 6-4. 해석 한계와 추가 검증

- 관찰 데이터이므로 결과는 “힌트 열람과 재발신 사이에 연관성이 있다”로 표현하며 인과를 단정하지 않는다.
- G2와 G1의 학교 규모, 학급 규모, 학년, 학교 유형, 성별, 가입일 분포가 다른지 먼저 확인한다.
- 분포 차이가 크면 해당 변수로 층화 비교하거나 성향점수 매칭을 실시한다.

## 현재까지의 결론

1. 전체 활성화 유저의 D+10 누적 재발신율은 67.4%이며, 재발신은 가입 초기에 집중되는 경향을 보였다.
2. 학교·학급 가입자 규모가 큰 구간일수록 D+7 재발신율이 대체로 높게 나타났다.
3. 학년과 학교 유형에 따른 재발신율 차이가 확인되어, 힌트 경험 비교 시 교란 가능성을 추가로 확인해야 한다.
4. 전체 활성화·미활성화 사용자의 단순 비교는 그룹의 출발선이 달라 대표 가설 검증에 사용하지 않았다.
5. 대표 가설 검증을 위해 가입 당일 PING을 발신한 사용자를 G0·G1·G2로 구분하고, 공통으로 D0 수신을 경험한 G2와 G1을 비교했다.
6. G2의 누적 재발신율은 G1보다 D+1에서 9.28%p, D+3에서 6.85%p, D+7에서 5.76%p, D+10에서 5.76%p 높았다.
7. 모든 관찰 시점에서 차이의 방향이 일관됐고 통계적으로 유의했으므로, “가입 당일 힌트를 열람한 사용자는 미열람 사용자보다 D+7 후속 PING 발신율이 높을 것이다”라는 대표 가설은 지지되었다.